In [49]:
import torch

In [50]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

In [51]:
# Simple attention for one input

query = inputs[1]

# Dot product over the vectors
attention_scores = torch.stack([torch.dot(query, input) for input in inputs])

print(attention_scores)

# Normailization(unstable, not good)
attention_scores_norm = attention_scores / attention_scores.sum()

print(attention_scores_norm)
print(f"Sum: {attention_scores_norm.sum()}")

# Normalization with softmax
attention_scores_norm = torch.softmax(attention_scores, dim=0)

print(attention_scores_norm)
print(f"Sum: {attention_scores_norm.sum()}")

# Calculate context vector of the query
context_vector = attention_scores_norm @ inputs
print(context_vector)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: 1.0000001192092896
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0
tensor([0.4419, 0.6515, 0.5683])


In [52]:
# Calculate attention for all inputs

# Dot product among single query
atten_scores = inputs @ inputs.T
print(atten_scores)

# Convert to weights
atten_weights = torch.softmax(atten_scores, dim=-1)
print(atten_weights)
print("all rows sum: ", atten_weights.sum(dim=-1))

# Context vector is all you get
ctx_vector = atten_weights @ inputs
print(ctx_vector)

assert torch.equal(ctx_vector[1], context_vector)  # same as calculated by hand

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
all rows sum:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [53]:
# Simple q,k,v for one input
x_2 = inputs[1]
print(x_2)

d_in = inputs.shape[1]
d_out = 2

assert x_2.shape[0] == d_in  # so that can be multiplied

torch.manual_seed(123)
W_q = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_k = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_v = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# q k v for query
query_2 = query @ W_q
key_2 = query @ W_k
value_2 = query @ W_v
print(query_2)

tensor([0.5500, 0.8700, 0.6600])
tensor([0.4306, 1.4551])


In [54]:
# k v for all
keys = inputs @ W_k
values = inputs @ W_v
print("keys shape", keys.shape)
print("values shape", values.shape)
print("projected input token into embedding space of ", d_out)

# attention scores for query_2
atten_scores_2 = query_2 @ keys.T
print(atten_scores_2)

# normalization
d_k = keys.shape[-1]
atten_weights_2 = torch.softmax(atten_scores_2 / d_k**0.5, dim=-1)
print("atten weights 2:", atten_weights_2)

# context vector
context_vector_2 = atten_weights_2 @ values
print(context_vector_2)

keys shape torch.Size([6, 2])
values shape torch.Size([6, 2])
projected input token into embedding space of  2
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
atten weights 2: tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor([0.3061, 0.8210])


In [55]:
# Attention class
import torch.nn as nn


class SelfAttention_V1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        self.d_k = self.W_key.shape[-1]

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        atten_scores = queries @ keys.T
        atten_weights = torch.softmax(atten_scores / (self.d_k**0.5), dim=-1)
        context_vector = atten_weights @ values

        return context_vector

In [56]:
torch.manual_seed(123)
att_v1 = SelfAttention_V1(d_in, d_out)
print(att_v1(inputs))

assert torch.equal(att_v1(inputs)[1], context_vector_2)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [57]:
# Attention class with nn.Linear
import torch.nn as nn


class SelfAttention_V2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        d_k = keys.shape[-1]
        atten_scores = queries @ keys.T
        atten_weights = torch.softmax(atten_scores / (d_k**0.5), dim=-1)
        context_vector = atten_weights @ values

        return context_vector

In [58]:
torch.manual_seed(789)
att_v2 = SelfAttention_V2(d_in, d_out)
print(att_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [59]:
# Causal attention
queries = att_v2.W_query(inputs)
keys = att_v2.W_key(inputs)
atten_scores = queries @ keys.T
atten_weights = torch.softmax(atten_scores / (keys.shape[-1] ** 0.5), dim=-1)
print(atten_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [60]:
context_length = atten_scores.shape[0]
mask = torch.tril(torch.ones(context_length, context_length))
print(mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [61]:
# element-wise, keep only lower triangular part
masked_weights = mask * atten_weights
print(masked_weights)

row_sum = masked_weights.sum(dim=-1, keepdim=True)
masked_weights = masked_weights / row_sum
print(masked_weights)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [62]:
# Mask out original scores, make it more efficient
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked_scores = atten_scores.masked_fill(mask.bool(), -torch.inf)
print(masked_scores)

masked_weights = torch.softmax(masked_scores / (keys.shape[-1] ** 0.5), dim=-1)
print(masked_weights)  # same result as the row_sum version

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
# Drop out
torch.manual_seed(123)
dropout_prob = torch.nn.Dropout(0.5)  # dropout rate of 50%
example = torch.ones(6, 6)  # create a matrix of ones

print(dropout_prob(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [64]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [ ]:
# Causal attention class
import torch
import torch.nn as nn


class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout_prob, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop_out = nn.Dropout(dropout_prob)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape
        queries = self.W_query(x)  # batch_size, num_token, d_out
        keys = self.W_key(x)
        values = self.W_value(x)

        atten_scores = queries @ keys.transpose(1, 2)
        atten_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        atten_weights = torch.softmax(atten_scores / (keys.shape[-1] ** 0.5), dim=-1)

        context_vector = atten_weights @ values
        return context_vector

In [66]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [67]:
# Multihead attention wrapper
import torch.nn as nn


class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout_prob, num_heads, qkv_bias=False) -> None:
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout_prob, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        head_outputs = [head(x) for head in self.heads]
        # Concatenate along the last dimension
        multihead_output = torch.cat(head_outputs, dim=-1)
        return multihead_output

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]  # number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vectors = mha(batch)
print(context_vectors)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)


In [ ]:
# Multihead Attention Class
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout_prob, num_heads, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # combine head outputs

        self.dropout = nn.Dropout(dropout_prob)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        # Step 1: Project to QKV
        # (2, 6, 3) → (2, 6, 4)
        # Each token goes from `d_in` to `d_out`
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Step 2: Split `d_out` into `num_heads * head_dim`
        # (2, 6, 4) → (2, 6, 2, 2)
        #!!! Now each head can process all tokens independently
        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim)

        # Step 3: Transpose to group heads together
        # (2, 6, 2, 2) → (2, 2, 6, 2)
        # (batch, token, head, dim) → (batch, head, token, dim)
        # Each head can independently process all tokens (token, dim)
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Step 4: Attention Computation
        # (2, 2, 6, 2) @ (2, 2, 2, 6) → (2, 2, 6, 6)
        atten_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        atten_scores = atten_scores.masked_fill_(mask_bool, -torch.inf)
        atten_weights = torch.softmax(atten_scores / keys.shape[-1] ** 0.5, dim=-1)

        # drop out weights
        atten_weights = self.dropout(atten_weights)

        # Step 5: Combine Heads Back
        # (2, 2, 6, 2) → (2, 6, 2, 2) -> [batch, tokens, heads, dims]
        context_vec = (atten_weights @ values).transpose(1, 2)

        # (2, 6, 2, 2) → (2, 6, 4) - merge heads back into single dimension
        context_vec = context_vec.contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.out_proj(context_vec)
        return context_vec

I'll explain the forward function step by step, focusing on the `view` and `transpose` operations which reshape the tensors for multi-head attention.

## Forward Function Breakdown

Let me use a concrete example with the dimensions from your code:
- `batch_size = 2`
- `num_tokens = 6` 
- `d_in = 3`
- `d_out = 4`
- `num_heads = 2`
- `head_dim = d_out // num_heads = 2`

### Step 1: Initial Projection (Lines 290-293)

```python
queries = self.W_query(x)  # (2, 6, 3) → (2, 6, 4)
keys = self.W_key(x)
values = self.W_value(x)
```

Each token's embedding goes from `d_in=3` to `d_out=4` dimensions.

### Step 2: Split into Heads with `.view()` (Lines 296-299)

```python
queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
# (2, 6, 4) → (2, 6, 2, 2)
```

**What `.view()` does:** It reshapes the tensor without changing the data order. Think of it as taking the last dimension (size 4) and splitting it into two parts (2 × 2).

**Why?** The `d_out=4` dimensions are really meant for 2 heads, each processing 2 dimensions. So we're making this explicit:
- Before: `[batch, tokens, 4]` - a flat 4D vector per token
- After: `[batch, tokens, 2 heads, 2 dims per head]` - explicitly separated into 2 heads

### Step 3: Transpose with `.transpose(1, 2)` (Lines 305-307)

```python
queries = queries.transpose(1, 2)
# (2, 6, 2, 2) → (2, 2, 6, 2)
# (batch, token, head, dim) → (batch, head, token, dim)
```

**What `.transpose(1, 2)` does:** It swaps dimensions 1 and 2 (0-indexed), moving heads before tokens.

**Why?** For parallel computation! Now the shape is `[batch, heads, tokens, dims]`:
- Each head can independently process all tokens
- You can compute attention for all heads simultaneously using batch matrix multiplication
- Head 0 processes all 6 tokens with its 2 dimensions
- Head 1 processes all 6 tokens with its 2 dimensions

### Step 4: Attention Computation (Lines 310-313)

```python
atten_scores = queries @ keys.transpose(2, 3)
# (2, 2, 6, 2) @ (2, 2, 2, 6) → (2, 2, 6, 6)
```

The matrix multiplication happens for each head independently:
- Shape: `[batch, heads, tokens, tokens]`
- For each batch and each head, you get a `(6×6)` attention score matrix

### Step 5: Combine Heads Back (Lines 318-320)

```python
context_vec = (atten_weights @ values).transpose(1, 2)
# (2, 2, 6, 2) → (2, 6, 2, 2) - swap heads and tokens back

context_vec = context_vec.contiguous().view(batch_size, num_tokens, self.d_out)
# (2, 6, 2, 2) → (2, 6, 4) - merge heads back into single dimension
```

**`.transpose(1, 2)`** swaps heads and tokens back to `[batch, tokens, heads, dims]`

**`.view()`** flattens the last two dimensions (2 heads × 2 dims) back into a single `d_out=4` dimension

**`.contiguous()`** is needed because `transpose` doesn't actually move data in memory—it just changes the view. `.contiguous()` creates a proper contiguous tensor so `.view()` can work.

## Visual Summary

```
Input: (batch=2, tokens=6, d_in=3)
    ↓ Linear projection
(2, 6, 4) ← single 4D vector per token
    ↓ .view() - split last dim
(2, 6, 2, 2) ← explicitly 2 heads × 2 dims
    ↓ .transpose(1,2) - swap tokens and heads
(2, 2, 6, 2) ← [batch, heads, tokens, head_dim]
    ↓ compute attention per head
(2, 2, 6, 2) ← context vectors per head
    ↓ .transpose(1,2) - swap back
(2, 6, 2, 2) ← [batch, tokens, heads, head_dim]
    ↓ .view() - merge heads
(2, 6, 4) ← single combined vector per token
    ↓ out_proj
Output: (2, 6, 4)
```

The key insight: **view splits/merges dimensions, transpose reorders them for parallel computation**.

In [79]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
